In [2]:
from master_llm import MasterLLM
from master_tokenizer import MasterTokenizer
import torch
import torch.nn as nn

In [ ]:
u_tokenizer = MasterTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)

In [4]:
torch.manual_seed(1)

u_model = MasterLLM(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings_with_attention_context = u_model(tokens)

sentence_meanings_with_attention_context

tensor([[ 0.7940, -0.4544,  0.4927,  0.3392],
        [ 0.5341, -0.2849,  0.2967,  0.1994],
        [ 0.1863, -0.5576,  0.3430,  0.2623],
        [ 0.0320, -0.5310,  0.2520,  0.2203],
        [ 0.0231, -0.3817,  0.2112,  0.1750],
        [ 0.0249, -0.3039,  0.1637,  0.1451],
        [ 0.0852, -0.1761,  0.1465,  0.1066]], grad_fn=<MmBackward0>)

![image.png](https://yqintl.alicdn.com/b5a4b0b0443864304af7e33822ad7788b19c4352.png)

In [5]:
#bilgi tutacak parametre
q_weights = torch.nn.Linear(4,3,bias=False)
k_weights = torch.nn.Linear(4,3,bias=False)
v_weights = torch.nn.Linear(4,3,bias=False)

q_of_sentences = q_weights(sentence_meanings_with_attention_context)
k_of_sentences = k_weights(sentence_meanings_with_attention_context)
v_of_sentences = v_weights(sentence_meanings_with_attention_context)

q_of_sentences.shape, k_of_sentences.shape, v_of_sentences.shape

(torch.Size([7, 3]), torch.Size([7, 3]), torch.Size([7, 3]))

In [6]:
attention_scores = q_of_sentences @ k_of_sentences.T
attention_weights = torch.softmax(attention_scores / k_of_sentences.shape[-1] ** 0.5, dim=-1)

context_vectors = attention_weights @ v_of_sentences
context_vectors

tensor([[ 0.1131,  0.3409, -0.2168],
        [ 0.1114,  0.3439, -0.2184],
        [ 0.1113,  0.3440, -0.2184],
        [ 0.1105,  0.3454, -0.2191],
        [ 0.1100,  0.3465, -0.2197],
        [ 0.1097,  0.3472, -0.2200],
        [ 0.1095,  0.3477, -0.2203]], grad_fn=<MmBackward0>)

In [7]:
from plot_tokens import plot_tokens

sentences = [
    {
        "words": q_of_sentences.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "blue",
    },
    {
        "words": k_of_sentences.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "purple", 
    },
    {
        "words": v_of_sentences.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "orange", 
    },
    {
        "words": context_vectors.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "green", 
    }
]
plot_tokens(sentences, "Query, Key and Value Vectors")

Casual Self Attention

In [8]:
attention_weights

torch.sum(attention_weights, dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)

![softmax](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRBCYFTne3DKjHYSa8BJoEqpUVXmThR6jWQwg&s)

In [9]:
# (N, N)
mask = torch.tril(torch.ones(attention_scores.shape[0], attention_scores.shape[0]))

# Mask uygula
masked_scores = attention_scores.masked_fill(mask == 0, -torch.inf)

# Softmax'tan sonra
softmaxed_attention_weights = torch.softmax(masked_scores, dim=1)

softmaxed_attention_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4764, 0.5236, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3001, 0.3305, 0.3694, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2242, 0.2414, 0.2615, 0.2729, 0.0000, 0.0000, 0.0000],
        [0.1818, 0.1920, 0.2044, 0.2111, 0.2107, 0.0000, 0.0000],
        [0.1535, 0.1602, 0.1685, 0.1728, 0.1725, 0.1726, 0.0000],
        [0.1330, 0.1377, 0.1439, 0.1470, 0.1466, 0.1466, 0.1452]],
       grad_fn=<SoftmaxBackward0>)

In [10]:
dropout_rate = 0.5
torch.manual_seed(1)

dropout = torch.nn.Dropout(dropout_rate)
dropout(softmaxed_attention_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6001, 0.6610, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3840, 0.4088, 0.4221, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.0000, 0.3369, 0.3456, 0.3450, 0.3452, 0.0000],
        [0.2661, 0.2753, 0.2878, 0.2939, 0.0000, 0.2932, 0.2903]],
       grad_fn=<MulBackward0>)

In [11]:
sentence_meanings_with_attention_context = u_model(tokens)

sentence_meanings_with_attention_context

tensor([[ 0.7940, -0.4544,  0.4927,  0.3392],
        [ 0.5341, -0.2849,  0.2967,  0.1994],
        [ 0.1863, -0.5576,  0.3430,  0.2623],
        [ 0.0320, -0.5310,  0.2520,  0.2203],
        [ 0.0231, -0.3817,  0.2112,  0.1750],
        [ 0.0249, -0.3039,  0.1637,  0.1451],
        [ 0.0852, -0.1761,  0.1465,  0.1066]], grad_fn=<MmBackward0>)

MultiHead Attention

In [16]:
from master_causal_attention import MasterCausalAttention

class MasterMultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, output_dim, context_length, num_heads, dropout_rate = 0):
        super().__init__()

        self.heads = nn.ModuleList(
        [MasterCausalAttention(embedding_dim, output_dim, context_length, dropout_rate) for _ in range(num_heads)]
        )

    def forward(self, x):
        attention_outs = []
        for head in self.heads:
            head_out = head(x)
            attention_outs.append(head_out)

        return torch.cat(attention_outs, dim=1)
        
multi_head_attention = MasterMultiHeadAttention(4, 4, 32, 2, dropout_rate=0)

out = multi_head_attention(torch.randn(4, 4))
out.shape, out

(torch.Size([4, 8]),
 tensor([[-0.7068,  0.2530,  0.8701, -0.5977, -0.9262, -0.6637, -0.5825, -0.2196],
         [-0.3143,  0.2317,  0.2587, -0.1868, -0.4504, -0.4146, -0.2770, -0.1345],
         [-0.4658,  0.0142,  0.6464, -0.2090, -0.1627, -0.3329, -0.1849, -0.0446],
         [-0.3538,  0.0842,  0.3491, -0.0211,  0.0526, -0.2867,  0.0197,  0.0471]],
        grad_fn=<CatBackward0>))